# Chat Models

interpkit's input layer accepts both plain strings and lists of chat messages (`[{"role": "user", "content": "..."}, ...]`). When you pass a message list, the tokenizer's chat template is applied automatically.

This notebook walks through:

1. Loading a small instruction-tuned model.
2. `model.chat(...)` — a one-line round-trip helper that returns the templated prompt and the generated response.
3. Running every existing op (`dla`, `scan`, `attention`, `steer`, ...) on chat-formatted prompts — no per-op chat code path required.

In [1]:
import interpkit

model = interpkit.load("HuggingFaceTB/SmolLM2-360M-Instruct")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

## Inspect the architecture

Despite being chat-tuned, SmolLM2 is just a standard Llama-style decoder under the hood. Every interpkit op that works on a base LM works here too.

In [2]:
model.inspect()

Model Architecture ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

LlamaForCausalLM · 32 layers · hidden=960 · vocab=49152 · 361.8M params total

Module                                     Type                   Params   Output Shape    Role    
 ────────────────────────────────────────────────────────────────────────────────────────────────── 
 model                                      LlamaModel                                              
 model.embed_tokens                         Embedding               47.2M   (1, 1, 960)      embed  
 model.layers                               ModuleList                                              
 model.layers.0                             LlamaDecoderLayer               (1, 1, 960)             
 model.layers.0.self_attn                   LlamaAttention                  (1, 1, 960)      attn   
 model.layers.0.self_attn.q_proj            Linear                 921.6K   (1, 1, 960)      attn   
 model.layers.0.self_attn.k_proj            Linear                 307.2K   (1, 1, 320)      attn   
 model.layers.0.self_attn.v_proj            Linear                 307.2K   (1, 1, 320)      attn   
 model.layers.0.self_attn.o_proj            Linear                 921.6K   (1, 1, 960)      attn   
 model.layers.0.mlp                         LlamaMLP                        (1, 1, 960)      mlp    
 model.layers.0.mlp.gate_proj               Linear                   2.5M   (1, 1, 2560)     mlp    
 model.layers.0.mlp.up_proj                 Linear                   2.5M   (1, 1, 2560)     mlp    
 model.layers.0.mlp.down_proj               Linear                   2.5M   (1, 1, 960)      mlp    
 model.layers.0.mlp.act_fn                  SiLUActivation                  (1, 1, 2560)     mlp    
 model.layers.0.input_layernorm             LlamaRMSNorm              960   (1, 1, 960)      norm   
 model.layers.0.post_attention_layernorm    LlamaRMSNorm              960   (1, 1, 960)      norm   
 model.layers.1                             LlamaDecoderLayer               (1, 1, 960)             
 model.layers.1.self_attn                   LlamaAttention                  (1, 1, 960)      attn   
 model.layers.1.self_attn.q_proj            Linear                 921.6K   (1, 1, 960)      attn   
 model.layers.1.self_attn.k_proj            Linear                 307.2K   (1, 1, 320)      attn   
 model.layers.1.self_attn.v_proj            Linear                 307.2K   (1, 1, 320)      attn   
 model.layers.1.self_attn.o_proj            Linear                 921.6K   (1, 1, 960)      attn   
 model.layers.1.mlp                         LlamaMLP                        (1, 1, 960)      mlp    
 model.layers.1.mlp.gate_proj               Linear                   2.5M   (1, 1, 2560)     mlp    
 model.layers.1.mlp.up_proj                 Linear                   2.5M   (1, 1, 2560)     mlp    
 model.layers.1.mlp.down_proj               Linear                   2.5M   (1, 1, 960)      mlp    
 model.layers.1.mlp.act_fn                  SiLUActivation                  (1, 1, 2560)     mlp    
 model.layers.1.input_layernorm             LlamaRMSNorm              960   (1, 1, 960)      norm   
 model.layers.1.post_attention_layernorm    LlamaRMSNorm              960   (1, 1, 960)      norm   
 model.layers.2                             LlamaDecoderLayer               (1, 1, 960)             
 model.layers.2.self_attn                   LlamaAttention                  (1, 1, 960)      attn   
 model.layers.2.self_attn.q_proj            Linear                 921.6K   (1, 1, 960)      attn   
 model.layers.2.self_attn.k_proj            Linear                 307.2K   (1, 1, 320)      attn   
 model.layers.2.self_attn.v_proj            Linear                 307.2K   (1, 1, 320)      attn   
 model.layers.2.self_attn.o_proj            Linear                 921.6K   (1, 1, 960)      attn   
 model.layers.2.mlp                         LlamaMLP                        (1, 1, 960)      mlp    
 model.layers.2.mlp.gate_proj               Linear                   2.5M   (1, 1, 2560)     mlp    
 model.layers.2.mlp.up_proj                 Linear            

## `model.chat()` — round-trip helper

Pass either a plain string (auto-wrapped as a single user turn) or a full list of messages. The result includes the templated prompt (so you can feed it to other ops) and the decoded response.

In [3]:
result = model.chat("Write a haiku about cats.", max_new_tokens=64)

print("=== prompt ===")
print(result["prompt"])
print("=== response ===")
print(result["response"])

=== prompt ===
<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Write a haiku about cats.<|im_end|>
<|im_start|>assistant

=== response ===
Whisker-waving, purring, and purring,
A feline's gentle presence, a gentle soul.
In the quiet of the night, a cat's grace,


### With a system prompt

Pass `system=...` for a one-shot system message, or build the message list yourself for full control.

In [4]:
model.chat(
    "What is the capital of France?",
    system="You are a terse geography assistant. Answer in one word.",
    max_new_tokens=16,
)

{'prompt': '<|im_start|>system\nYou are a terse geography assistant. Answer in one word.<|im_end|>\n<|im_start|>user\nWhat is the capital of France?<|im_end|>\n<|im_start|>assistant\n',
 'response': 'F.',
 'messages': [{'role': 'system',
   'content': 'You are a terse geography assistant. Answer in one word.'},
  {'role': 'user', 'content': 'What is the capital of France?'}],
 'input_ids': tensor([[    1,  9690,   198,  2683,   359,   253,  1733,   313, 12973, 11173,
             30, 19842,   281,   582,  2229,    30,     2,   198,     1,  4093,
            198,  1780,   314,   260,  3575,   282,  4649,    47,     2,   198,
              1,   520,  9531,   198]], device='mps:0'),
 'output_ids': tensor([[    1,  9690,   198,  2683,   359,   253,  1733,   313, 12973, 11173,
             30, 19842,   281,   582,  2229,    30,     2,   198,     1,  4093,
            198,  1780,   314,   260,  3575,   282,  4649,    47,     2,   198,
              1,   520,  9531,   198,    54,    30,     2

In [5]:
model.chat(
    [
        {"role": "system", "content": "You are a terse geography assistant."},
        {"role": "user", "content": "Capital of Germany?"},
        {"role": "assistant", "content": "Berlin."},
        {"role": "user", "content": "Capital of Italy?"},
    ],
    max_new_tokens=16,
)

{'prompt': '<|im_start|>system\nYou are a terse geography assistant.<|im_end|>\n<|im_start|>user\nCapital of Germany?<|im_end|>\n<|im_start|>assistant\nBerlin.<|im_end|>\n<|im_start|>user\nCapital of Italy?<|im_end|>\n<|im_start|>assistant\n',
 'response': 'Rome.',
 'messages': [{'role': 'system',
   'content': 'You are a terse geography assistant.'},
  {'role': 'user', 'content': 'Capital of Germany?'},
  {'role': 'assistant', 'content': 'Berlin.'},
  {'role': 'user', 'content': 'Capital of Italy?'}],
 'input_ids': tensor([[    1,  9690,   198,  2683,   359,   253,  1733,   313, 12973, 11173,
             30,     2,   198,     1,  4093,   198, 39832,   282,  4521,    47,
              2,   198,     1,   520,  9531,   198, 24726,  4226,    30,     2,
            198,     1,  4093,   198, 39832,   282,  7158,    47,     2,   198,
              1,   520,  9531,   198]], device='mps:0'),
 'output_ids': tensor([[    1,  9690,   198,  2683,   359,   253,  1733,   313, 12973, 11173,
        

## Running other ops on a chat prompt

Every interpkit op takes the same `input_data` argument, so you can pass either:

- The string `result["prompt"]` returned by `chat()`, or
- A list of message dicts directly — `prepare_input` will template it.

Both forms produce the same `input_ids`.

In [6]:
result = model.chat("What is the capital of France?", max_new_tokens=8)

# DLA on the templated prompt
dla_out = model.dla(result["prompt"], top_k=5)
print(f"Top contributor to {dla_out['target_token']!r}: "
      f"{dla_out['contributions'][0]['component']}")

Direct Logit Attribution ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭────────────────────────────────────────────────────╮
│  Target token: "The"  |  Total logit sum: 234.862  │
╰────────────────────────────────────────────────────╯

 Component   Type   Contribution                               
 ────────────────────────────────────────────────────────────── 
  L30.mlp     mlp        +31.1873   ████████████████████████    
  L31.mlp     mlp        +27.8655   █████████████████████       
  L22.mlp     mlp        +27.3026   █████████████████████       
  L30.attn    attn       +19.4311   ██████████████▌             
  L31.attn    attn       +17.8552   █████████████▌              
  ...                               (59 more)

Per-Head Breakdown  (top 5)

 Head      Contribution                               
 ───────────────────────────────────────────────────── 
  L31.H7        +21.5842   ████████████████████████    
  L30.H5        +14.7935   ████████████████            
  L31.H8         +6.5672   ███████                     
  L27.H13        +6.5396   ███████                     
  L24.H1         +5.6892   ██████                      
  L27.H2         -2.6236   ██▌                         
  L27.H14        -2.9113   ███                         
  L29.H6         -3.1907   ███▌                        
  L16.H11        -3.2543   ███▌                        
  L31.H6        -16.9993   ██████████████████▌

Top contributor to 'The': L30.mlp


In [7]:
# Pass messages directly to any op — prepare_input applies the chat template.
model.dla(
    [{"role": "user", "content": "What is the capital of France?"}],
    top_k=5,
)

Direct Logit Attribution ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭────────────────────────────────────────────────────╮
│  Target token: "The"  |  Total logit sum: 234.862  │
╰────────────────────────────────────────────────────╯

 Component   Type   Contribution                               
 ────────────────────────────────────────────────────────────── 
  L30.mlp     mlp        +31.1873   ████████████████████████    
  L31.mlp     mlp        +27.8655   █████████████████████       
  L22.mlp     mlp        +27.3026   █████████████████████       
  L30.attn    attn       +19.4311   ██████████████▌             
  L31.attn    attn       +17.8552   █████████████▌              
  ...                               (59 more)

Per-Head Breakdown  (top 5)

 Head      Contribution                               
 ───────────────────────────────────────────────────── 
  L31.H7        +21.5842   ████████████████████████    
  L30.H5        +14.7935   ████████████████            
  L31.H8         +6.5672   ███████                     
  L27.H13        +6.5396   ███████                     
  L24.H1         +5.6892   ██████                      
  L27.H2         -2.6236   ██▌                         
  L27.H14        -2.9113   ███                         
  L29.H6         -3.1907   ███▌                        
  L16.H11        -3.2543   ███▌                        
  L31.H6        -16.9993   ██████████████████▌

{'target_token': 'The',
 'target_id': 504,
 'contributions': [{'component': 'L30.mlp',
   'layer': 30,
   'type': 'mlp',
   'logit_contribution': 31.18732452392578,
   'module': 'model.layers.30'},
  {'component': 'L31.mlp',
   'layer': 31,
   'type': 'mlp',
   'logit_contribution': 27.865493774414062,
   'module': 'model.layers.31'},
  {'component': 'L22.mlp',
   'layer': 22,
   'type': 'mlp',
   'logit_contribution': 27.302574157714844,
   'module': 'model.layers.22'},
  {'component': 'L30.attn',
   'layer': 30,
   'type': 'attn',
   'logit_contribution': 19.431114196777344,
   'module': 'model.layers.30'},
  {'component': 'L31.attn',
   'layer': 31,
   'type': 'attn',
   'logit_contribution': 17.85519790649414,
   'module': 'model.layers.31'},
  {'component': 'L24.mlp',
   'layer': 24,
   'type': 'mlp',
   'logit_contribution': 16.825645446777344,
   'module': 'model.layers.24'},
  {'component': 'L26.mlp',
   'layer': 26,
   'type': 'mlp',
   'logit_contribution': 16.294361114501953

In [8]:
model.scan(result["prompt"])

▸ Running DLA...

Direct Logit Attribution ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭────────────────────────────────────────────────────╮
│  Target token: "The"  |  Total logit sum: 234.862  │
╰────────────────────────────────────────────────────╯

 Component   Type   Contribution                               
 ────────────────────────────────────────────────────────────── 
  L30.mlp     mlp        +31.1873   ████████████████████████    
  L31.mlp     mlp        +27.8655   █████████████████████       
  L22.mlp     mlp        +27.3026   █████████████████████       
  L30.attn    attn       +19.4311   ██████████████▌             
  L31.attn    attn       +17.8552   █████████████▌              
  ...                               (59 more)

Per-Head Breakdown  (top 5)

...

Predictions: "The" (28.5%), "I" (22.2%), "As" (5.6%)

   1   dla            Top contributor to "The": L30.mlp (+31.187)                
   2   dla            Top attention head: L31.H7 (+21.584)                       
   3   attention        → attends "<|im_start|>" → "<|im_start|>" (weight 1.00)  
   4   attribution    Most salient input token: "LM" (score 12.250)              
   5   attention      Most focused attention: L18.H9 (entropy 0.00)              
   6   prediction     Top prediction: "The" (28.5%)                              
   7   lens           Answer "The" first appears at layer 30/32 (model.layers.30)

╭───────────────── Top finding ─────────────────╮
│  Top contributor to "The": L30.mlp (+31.187)  │
╰───────────────────────────────────────────────╯

Analyses: dla · lens · attention · attribution  |  Input: "<|im_start|>system
You are a helpful AI assistant named S..."

{'input': '<|im_start|>system\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>\n<|im_start|>user\nWhat is the capital of France?<|im_end|>\n<|im_start|>assistant\n',
 'prediction': {'top5_tokens': ['The', 'I', 'As', 'You', 'Hello'],
  'top5_probs': [0.2845183312892914,
   0.22158309817314148,
   0.056024983525276184,
   0.0494418740272522,
   0.040988754481077194]},
 'dla': {'target_token': 'The',
  'target_id': 504,
  'contributions': [{'component': 'L30.mlp',
    'layer': 30,
    'type': 'mlp',
    'logit_contribution': 31.18732452392578,
    'module': 'model.layers.30'},
   {'component': 'L31.mlp',
    'layer': 31,
    'type': 'mlp',
    'logit_contribution': 27.865493774414062,
    'module': 'model.layers.31'},
   {'component': 'L22.mlp',
    'layer': 22,
    'type': 'mlp',
    'logit_contribution': 27.302574157714844,
    'module': 'model.layers.22'},
   {'component': 'L30.attn',
    'layer': 30,
    'type': 'attn',
    'logit_contribution': 19.43111

## Steering on chat inputs

`steer_vector` accepts message lists too. Below we extract a polite-vs-rude direction at a mid layer and apply it to a third chat input.

(SmolLM2's residual-stream module name is `model.layers.<i>` — `model.inspect()` above shows the full tree.)

In [9]:
polite = [
    [{"role": "user", "content": "Could you please summarise this politely?"}],
    [{"role": "user", "content": "Would you kindly help me with a task?"}],
]
rude = [
    [{"role": "user", "content": "Just summarise it. Now."}],
    [{"role": "user", "content": "Help me. Hurry up."}],
]

vector = model.steer_vector(polite, rude, at="model.layers.10")
print(f"Vector shape: {vector.shape}, norm: {vector.float().norm():.2f}")

Output()

Vector shape: torch.Size([960]), norm: 30.54


In [10]:
model.steer(
    [{"role": "user", "content": "Tell me about your weekend."}],
    vector=vector,
    at="model.layers.10",
    scale=4.0,
)

Steering — model.layers.10  scale=4.0 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

 Rank   Original Token    Prob         Steered Token    Prob  
 ───────────────────────────────────────────────────────────── 
     1   I                0.277    ·    I               0.142  
     2   My               0.191    →    As              0.134  
     3   As               0.062    →    My              0.134  
     4   Sure             0.040    ·    Sure            0.059  
     5   Ah               0.027    →    Yes             0.056  
     6   H                0.019    →    Well            0.034  
     7   The              0.015    →    Unfortunately   0.032  
     8   Hi               0.015    →    Saturday        0.019  
     9   It               0.014    →    The             0.017  
    10   Yes              0.014    →    While           0.016

{'original_logits': tensor([[[ 9.8125, 15.2500, 19.0000,  ...,  6.3750,  0.6133,  4.9062],
          [ 8.6250,  9.3750,  9.8125,  ..., -4.2500, -9.3125, -8.5625],
          [ 0.8789, 17.7500, 21.5000,  ..., -6.7500, -6.0000, -1.9922],
          ...,
          [ 2.5781, -1.4297, -0.8438,  ..., -1.5312, -3.5938, -3.7500],
          [ 8.3125, 10.0625, 15.6875,  ...,  0.3105, -3.9844, -0.2188],
          [ 2.3125, -5.0938, -3.3438,  ..., -2.6562, -2.0938, -6.7812]]],
        device='mps:0', dtype=torch.bfloat16),
 'steered_logits': tensor([[[10.1250, 15.1250, 19.0000,  ...,  6.4375,  0.6758,  4.9375],
          [ 7.5312,  2.8281,  4.7812,  ..., -2.2031, -3.8594, -9.0000],
          [ 2.9688,  6.2812,  9.3125,  ..., -7.5938, -6.7500, -2.7031],
          ...,
          [ 2.9062, -1.1953, -1.3906,  ..., -4.1250, -5.4062, -3.3125],
          [ 9.7500,  6.6250, 11.1875,  ...,  4.2500, -2.0625, -1.2578],
          [ 1.3125, -4.9688, -3.2344,  ..., -4.4062, -3.9375, -8.6875]]],
        device='mp

## Caveats

- **Chat templates differ between model families.** The Llama, Qwen, ChatML, Gemma, and Mistral families each ship their own template. interpkit just calls whatever `tokenizer.apply_chat_template` is configured for the loaded model.
- **`add_generation_prompt=True` is always set** when interpkit applies a template. This appends the assistant marker so the model knows to start producing a reply. Without it, the model often continues the *user* turn instead.
- **Base models without a template raise `ValueError`.** If you load a non-instruct model and pass a message list, you'll get a clear error suggesting you pass a plain string or load a chat variant.
- **Sampling defaults are greedy.** `chat(do_sample=False)` is the default; pass `do_sample=True` plus `temperature` / `top_p` for stochastic generation.

## From the CLI

Everything above also works from the command line. The `chat` subcommand applies the tokenizer's chat template, calls `model.generate`, and prints the response in a styled panel.

```bash
# Basic usage
interpkit chat HuggingFaceTB/SmolLM2-360M-Instruct "Write a haiku about cats." --max-new-tokens 64

# With a system prompt and the templated prompt printed for inspection
interpkit chat HuggingFaceTB/SmolLM2-360M-Instruct \
    "What is the capital of France?" \
    --system "You are a terse geography assistant." \
    --show-prompt

# Sampling instead of greedy
interpkit chat HuggingFaceTB/SmolLM2-360M-Instruct "Tell me a joke." \
    --sample --temperature 0.8 --top-p 0.9
```

If the `interpkit` console script isn't on your `PATH` (fresh env, sandboxed install, ...), `python -m interpkit` works the same:

```bash
python -m interpkit chat HuggingFaceTB/SmolLM2-360M-Instruct "Hello!"
```

Run `interpkit chat --help` for the full option list, or `interpkit --extensive` for a beginner-friendly walkthrough of every command.